# Trace figure: demo on the running example

Concept: `overview/Single Level Visit Mappings.md` (outside this repo); the figure is a draft for discussing visits and not part of the tool yet.
Code: modules `ocelescope_module_location_based_ocpm.visits.trace_figure` and `visits.context`; this notebook only imports and calls their functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(trace_figure)`) and `docs/resources/trace-figure.md`.

What the notebook does:
1. loads the location mapping of one level (saved by `location_derivation.ipynb`) and the location context ($agg$, $V$, $W$),
2. prepares the relations for the figure step by step (high-level locations, location kinds, x positions, high-level runs),
3. runs the same with one call (`relations_for_figure`),
4. draws the trace figure,
5. shows the three x axes,
6. adds visits as bars (hand-written illustration; visits are not computed yet),
7. saves the figure as an HTML file.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/trace_figure.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$.

In [ ]:
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import load_level_mapping
from ocelescope_module_location_based_ocpm.visits.context import location_context
from ocelescope_module_location_based_ocpm.visits.trace_figure import (
    add_high_level_locations,
    add_location_kinds,
    add_x_positions,
    high_level_runs,
    relations_for_figure,
    trace_figure,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
DERIVED = EXAMPLE / "derived"
OUTPUT = ROOT / "notebooks" / "output"

# Show whole frames (the running example has 28 relations).
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250);

## 1. Inputs

### Location mapping: `load_level_mapping(folder, level)`

The current locations of one level, as saved by `location_derivation.ipynb` (section 8) in `data/running_example/derived/locations_level_{k}.csv`.

| parameter | meaning |
|---|---|
| `folder` | folder of the saved files |
| `level` | level $k$ (0 to 3) |

Returns one row per hu-event relation, sorted by hu, time, event, with columns event, hu, object_type, time, activity, is_movement, is_transformation, `loc` ($= loc_k$) and `ploc` ($= ploc_k$).

Here: Level 3, the selected level of the running example.

In [ ]:
LEVEL = 3
mapping = load_level_mapping(DERIVED, LEVEL)
mapping

### Location context: `location_context(table, location, high_level, is_vehicle, is_storage)`

The location mapping $agg: L \not\to HL$, the vehicles $V$ and the storage units $W$, read from a table with one row per location.
For the running example: `data/running_example/inputs/locations.csv` (written by `make_running_example.py`; the input format for visits is a draft until it is decided).

| parameter | meaning | default |
|---|---|---|
| `table` | the table (polars frame) | |
| `location` | column of the low-level location $l$ | `"location"` |
| `high_level` | column of $agg(l)$; an empty cell leaves $l$ unmapped | `"high_level"` |
| `is_vehicle` | boolean column: $l \in V$ (missing column = no vehicles) | `"is_vehicle"` |
| `is_storage` | boolean column: $l \in W$ (missing column = no storage units) | `"is_storage"` |

Returns a `LocationContext` with `agg` (dict, only the mapped locations), `vehicles` and `storage_units` (sets).

Example: A1 and A2 map to A, B1 to B; T is not mapped, so it is its own high-level location; $V = \{T\}$, $W = \{B1\}$.

In [ ]:
locations = pl.read_csv(EXAMPLE / "inputs" / "locations.csv")
locations

In [ ]:
context = location_context(locations)
context

## 2. Preparing the relations, step by step

Each step takes a relations frame and returns it with one or more columns added.
Here they are applied to the handling units in the order in which the figure shows them (top to bottom).

### High-level locations: `add_high_level_locations(relations, location, agg, high_level="high_level")`

$\widehat{hl}((e, hu)) = agg(loc((e, hu)))$ if $loc((e, hu)) \ne \bot$, else $\bot$.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with the column `location` | |
| `location` | column of $loc$ | |
| `agg` | dict low-level → high-level location; locations it does not map are their own high-level location (identity) | identity for all |
| `high_level` | name of the new column | `"high_level"` |

How it works: a polars `replace` on the location column; values not in `agg` stay as they are, and null stays null.

Example: c1's relations map to A, A, A, B, T, A, A, T, T.

In [ ]:
HUS = ["s1", "c1", "c2", "c3", "c4", "c5"]

rel = mapping.filter(pl.col("hu").is_in(HUS))
rel = add_high_level_locations(rel, "loc", context.agg)
rel.select("hu", "event", "time", "activity", "loc", "high_level")

### Location kinds: `add_location_kinds(relations, location, vehicles=(), storage_units=(), kind="location_kind")`

Whether a relation's low-level location is a vehicle, a storage unit or neither; the figure draws vehicles as open circles and storage units as diamonds.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with the column `location` | |
| `location` | column of $loc$ | |
| `vehicles` | the set $V$ | empty |
| `storage_units` | the set $W$ | empty |
| `kind` | name of the new column | `"location_kind"` |

Values: `vehicle`, `storage`, `other`, and null for $\bot$.
A location in both $V$ and $W$ (violating RQ4) counts as a vehicle.

In [ ]:
rel = add_location_kinds(rel, "loc", context.vehicles, context.storage_units)
rel.select("hu", "event", "loc", "high_level", "location_kind")

### Positions: `add_x_positions(relations, axis="event order")`

Adds `position`, the index of the relation in $otrace(hu)$ (ordered by time, then event id; starting at 1), and `x`, the relation's x value in the figure:

| `axis` | x | use |
|---|---|---|
| `"event order"` (default) | rank of the event among all shown events | an event shared by several handling units has one x, so shared events line up |
| `"trace position"` | `position` | the position the visit definition counts in |
| `"time"` | the timestamp | real durations |

Handling units are ordered by the column `hu_order` if present, else by id.
Example (event order): e3 has x = 3 for s1, c1 and c2; c1's e3 is at trace position 1.

In [ ]:
rel = add_x_positions(rel.with_columns(pl.col("hu").replace_strict({hu: i for i, hu in enumerate(HUS)}).alias("hu_order")))
rel.select("hu", "event", "time", "high_level", "position", "x")

### High-level runs: `high_level_runs(relations)`

The runs of consecutive relations of a handling unit with the same high-level location; the figure draws one ribbon per run.
Runs of $\bot$ are left out.
Returns one row per run: hu, high_level, x0 (x of its first relation), x1 (x of its last relation).

How it works: a run starts where `high_level` differs from the previous relation of the same handling unit; a cumulative sum of these starts numbers the runs.

Example: c1 has the runs A, B, T, A, T.

In [ ]:
high_level_runs(rel)

## 3. All in one call: `relations_for_figure(relations, location="loc", handling_units=None, agg=None, vehicles=(), storage_units=(), axis="event order")`

Selects the handling units and runs the first three steps.

| parameter | meaning | default |
|---|---|---|
| `relations` | one row per hu-event relation, e.g. a loaded level mapping or `Derivation.relations` | |
| `location` | column of $loc$ (e.g. `"loc_3"` on `Derivation.relations`) | `"loc"` |
| `handling_units` | handling units to show, top to bottom in this order | all, sorted by id |
| `agg`, `vehicles`, `storage_units` | the location context | identity, none, none |
| `axis` | see `add_x_positions` | `"event order"` |

Returns columns hu, hu_order, object_type, event, time, activity, is_movement, is_transformation, loc, high_level, location_kind, position, x.
The runs are computed inside `trace_figure`.

In [ ]:
relations = relations_for_figure(mapping, "loc", HUS, context.agg, context.vehicles, context.storage_units)
relations.equals(rel.select(relations.columns))

## 4. The figure: `trace_figure(relations, visits=None, filled_groups=("resolved",), dashed_groups=("maximal",), high_level_style="ribbon", show_event_labels=True, link_shared_events=True, mark_ties=True, show_visit_ids=True, title=None, width=1200)`

| parameter | meaning | default |
|---|---|---|
| `relations` | output of `relations_for_figure` | |
| `visits` | optional table of visits, one row per visit (section 6) | none |
| `filled_groups` | visit groups drawn as filled bars; all others outlined | `("resolved",)` |
| `dashed_groups` | visit groups with a dashed outline (maximal visits, to tell them apart from resolved ones) | `("maximal",)` |
| `high_level_style` | `"ribbon"`: one labelled bar per high-level run below the trace (the other values are earlier alternatives) | `"ribbon"` |
| `show_event_labels` | event ids at an angle above the markers | `True` |
| `link_shared_events` | dashed lines joining an event shared by several handling units (not on the trace position axis) | `True` |
| `show_visit_ids` | write the `visit_id` of each visit (if the column is given) in the middle of its bar | `True` |
| `mark_ties` | light grey background box behind events with the same timestamp: over the whole figure on the event order and time axes, per handling unit on the trace position axis | `True` |
| `title` | figure title | none |
| `width` | figure width in pixels; a ribbon label is only shown if it fits at this width (otherwise legend and hover name the location) | 1200 |

How it works: per handling unit (y label: id and, below it, object type in brackets)
- the trace: a grey line through one marker per relation, coloured by $loc$ (circle; open circle for vehicles, diamond for storage units, grey × for $\bot$), with the event id above;
- the ribbons: one bar per high-level run, coloured by $\widehat{hl}$ and labelled with its name where it fits;
- the visit lanes (if `visits` is given), see section 6.

Low-level and high-level locations share one colour map, ranked by frequency in the figure (`location_colours`), so a location mapped to itself (T) has one colour in both.
Hover over a marker or ribbon for the event, activity, time, trace position and both locations.

In [ ]:
trace_figure(relations, title=f"Running example, Level {LEVEL}").show()

## 5. Axes

The same handling units with `axis="trace position"` (each trace starts at 1; no dashed lines, since a shared event has different x values) and `axis="time"` (gaps between events become visible; events with the same timestamp are spread slightly around it in event id order, so they do not hide each other, see `add_tie_offsets`; the hover shows the true time).

Events with the same timestamp get a light grey background box (`mark_ties`); the running example has no such events, see `docs/resources/trace-figure.md`.

In [ ]:
for axis in ("trace position", "time"):
    rel_axis = relations_for_figure(mapping, "loc", HUS, context.agg, context.vehicles, context.storage_units, axis=axis)
    trace_figure(rel_axis, title=f"x = {axis}").show()

## 6. Visits

Visits are not computed yet; `trace_figure` draws visits given as a table with one row per visit:

| column | meaning |
|---|---|
| `hu` | handling unit |
| `high_level` | high-level location of the visit; empty (null) for a visit at $\bot$ |
| `first_event`, `last_event` | first and last event of the visit (its infix of $otrace(hu)$) |
| `visit_id` | optional id, written in the middle of the bar and in the hover (`show_visit_ids=False` hides it); makes it easier to talk about single visits |
| `group` | visit group, e.g. "maximal" or "resolved"; each group gets its own lanes, labelled with its name |

How it works: each visit becomes a bar from its first to its last relation, coloured by its location (grey for $\bot$); within a group, visits sharing relations go into different lanes (`visit_lanes`, greedy interval packing, containing visits above contained ones), so contained and overlapping visits are visible.
Groups in `filled_groups` are filled, all others outlined; groups in `dashed_groups` (default: maximal) have a dashed outline.

Below: the maximal visits at κ = 2, written by hand for illustration (c1: A from e3 to e11 overlaps T from e8 to e14 and contains B at e7; c2: A from e3 to e11 contains T at e8; c5: its only visit is at $\bot$, e16 to e17, since $\kappa(\bot) = 0$ and both of its relations are not located).

In [ ]:
VISITS_K2 = pl.DataFrame(
    [
        ("m1", "s1", "A", "e1", "e3"),
        ("m2", "c1", "A", "e3", "e11"),
        ("m3", "c1", "B", "e7", "e7"),
        ("m4", "c1", "T", "e8", "e14"),
        ("m5", "c2", "A", "e3", "e11"),
        ("m6", "c2", "T", "e8", "e8"),
        ("m7", "c2", "B", "e12", "e15"),
        ("m8", "c3", "T", "e8", "e8"),
        ("m9", "c3", "B", "e10", "e14"),
        ("m10", "c4", "B", "e15", "e16"),
        ("m11", "c5", None, "e16", "e17"),
    ],
    schema=["visit_id", "hu", "high_level", "first_event", "last_event"],
    orient="row",
).with_columns(pl.lit("maximal").alias("group"))
VISITS_K2

In [ ]:
figure = trace_figure(relations, VISITS_K2, title="Maximal visits at κ = 2 (illustration)")
figure.show()

### Resolved visits

A second group, `"resolved"`, is drawn as filled bars below the maximal ones (`filled_groups` default).
Resolving is not defined yet either; the table below is a hand-written illustration in which every relation (including the not located ones) belongs to exactly one visit, using two example rules:
- **contained**: the outer visit wins, the contained visit dissolves into its excursion (c1: B at e7; c2: T at e8);
- **overlapping**: the earlier visit wins, the later one starts after it ends (c1: T now starts at e13 instead of e8).

All other maximal visits stay as they are.

In [ ]:
VISITS_RESOLVED = pl.DataFrame(
    [
        ("r1", "s1", "A", "e1", "e3"),
        ("r2", "c1", "A", "e3", "e11"),
        ("r3", "c1", "T", "e13", "e14"),
        ("r4", "c2", "A", "e3", "e11"),
        ("r5", "c2", "B", "e12", "e15"),
        ("r6", "c3", "T", "e8", "e8"),
        ("r7", "c3", "B", "e10", "e14"),
        ("r8", "c4", "B", "e15", "e16"),
        ("r9", "c5", None, "e16", "e17"),
    ],
    schema=["visit_id", "hu", "high_level", "first_event", "last_event"],
    orient="row",
).with_columns(pl.lit("resolved").alias("group"))

figure = trace_figure(
    relations,
    pl.concat([VISITS_K2, VISITS_RESOLVED]),
    title="Maximal and resolved visits at κ = 2 (illustration)",
)
figure.show()

## 7. Save the figure

`figure.write_html(path)` saves an interactive file; with `include_plotlyjs="cdn"` the file is small but needs internet to open.
The files go to `notebooks/output/` (not committed).

In [ ]:
OUTPUT.mkdir(exist_ok=True)
figure.write_html(OUTPUT / "trace-figure.html", include_plotlyjs="cdn")